# Incertidumbre por variabilidad

## Objetivo del notebook
Aprender a medir **incertidumbre caso a caso** a partir de la **variabilidad**
en las predicciones del modelo.

Aquí no solo importa *qué probabilidad* se predice,
sino **qué tan estable es esa predicción** frente a pequeñas variaciones.

-

## Idea central
Si el modelo:
- cambia mucho de opinión ante pequeñas perturbaciones, o
- distintos modelos entrenados de forma similar discrepan,

entonces existe **incertidumbre**, incluso si la probabilidad es alta.

En otras palabras:
> alta probabilidad no siempre implica alta certeza.

-

## Qué entendemos por variabilidad
Variabilidad es la **dispersión** de las probabilidades predichas
cuando repetimos la inferencia de formas ligeramente distintas, por ejemplo:
- usando varios modelos (ensemble),
- usando diferentes subconjuntos de datos,
- usando inicializaciones distintas.

Cuanta más variación observemos, **menos confiable** es la predicción.

-

## Alcance de este notebook
- Técnica principal: **Ensembles simples**
- Modelos base:
  - Regresión Logística
  - Árbol de Decisión
- Señales que construiremos:
  - media de probabilidades
  - varianza / desviación estándar
- Regla operativa:

---

## 2) Reconstrucción del experimento base y creación del ensemble

### ¿Qué vamos a hacer?
Vamos a reconstruir el mismo experimento base (dataset y splits)
y crear un **ensemble simple** entrenando varios modelos similares,
pero con pequeñas variaciones.

Cada modelo verá datos ligeramente distintos.
La **dispersión de sus predicciones** será nuestra señal de incertidumbre.

--

### ¿Por qué un ensemble?
Si varios modelos razonables:
- coinciden → alta confianza,
- discrepan → alta incertidumbre.

Esto nos permite medir duda **sin cambiar la arquitectura**.

In [7]:
# Imports
import numpy as np
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

In [8]:
# Dataset base (mismos parámetros)
X, y = make_classification(
    n_samples=4000,
    n_features=10,
    n_informative=5,
    n_redundant=2,
    n_clusters_per_class=2,
    class_sep=1.0,
    flip_y=0.05,
    random_state=42
)

# Split: train / test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

---

### 2.1) Crear un ensemble de Regresión Logística

Entrenamos varios modelos iguales,
pero cada uno ve una muestra bootstrap distinta del training set.

In [9]:
from sklearn.utils import resample

N_MODELS = 10
lr_ensemble = []

for i in range(N_MODELS):
    X_boot, y_boot = resample(
        X_train, y_train,
        replace=True,
        random_state=42 + i
    )
    
    model = LogisticRegression(max_iter=1000, random_state=42 + i)
    model.fit(X_boot, y_boot)
    lr_ensemble.append(model)

Aquí tienes la explicación **paso a paso**, en lenguaje claro:

1. **Importa `resample`**

* `resample` sirve para crear una nueva muestra a partir de tus datos.
* Es como “rearmar” el dataset de entrenamiento eligiendo filas al azar.

2. **Define cuántos modelos vas a entrenar**

* `N_MODELS = 10` significa que vas a entrenar **10 regresiones logísticas**.

3. **Crea una lista vacía para guardar los modelos**

* `lr_ensemble = []` será el contenedor donde se guardan los 10 modelos entrenados.

4. **Bucle para entrenar los 10 modelos**

* `for i in range(N_MODELS):` repite el proceso 10 veces.

5. **Crea un “bootstrapped dataset” para cada modelo**

```python
X_boot, y_boot = resample(X_train, y_train, replace=True, random_state=42+i)
```

* Toma el conjunto de entrenamiento (`X_train`, `y_train`) y crea una versión nueva:

  * `replace=True` significa **con reemplazo**: una fila puede repetirse varias veces y otras pueden no aparecer.
  * `random_state=42+i` cambia la semilla en cada iteración, así cada modelo ve un dataset ligeramente distinto.

6. **Crea el modelo**

```python
model = LogisticRegression(max_iter=1000, random_state=42+i)
```

* Inicializa una Regresión Logística.
* `max_iter=1000` asegura que tenga suficientes iteraciones para converger.
* `random_state=42+i` hace que el comportamiento sea reproducible y ligeramente distinto por iteración.

7. **Entrena el modelo con su versión del dataset**

```python
model.fit(X_boot, y_boot)
```

* Entrena la regresión logística usando el dataset “remuestreado”.

8. **Guarda el modelo entrenado**

```python
lr_ensemble.append(model)
```

* Añade ese modelo a la lista.
* Al final, `lr_ensemble` contiene **10 modelos distintos**, entrenados con datos parecidos pero no idénticos.

✅ Resultado final:
Tienes un **ensemble**. Si esos 10 modelos dan probabilidades distintas para el mismo caso, eso es una señal de **incertidumbre por variabilidad**.

In [10]:
probs_lr_ensemble[:, :5]

array([[0.51795306, 0.62411521, 0.12093473, 0.32668206, 0.72072066],
       [0.5638592 , 0.67695723, 0.14014511, 0.25475264, 0.76709697],
       [0.57595658, 0.61619421, 0.11971005, 0.27387953, 0.69022418],
       [0.53383332, 0.68679604, 0.09619943, 0.31242495, 0.74710676],
       [0.58694646, 0.65353343, 0.14208567, 0.2753118 , 0.71830665],
       [0.56703054, 0.61797383, 0.12830362, 0.29161196, 0.69327374],
       [0.5702701 , 0.68537505, 0.14330071, 0.29908052, 0.7487849 ],
       [0.55884171, 0.6375322 , 0.12811269, 0.30670005, 0.71436494],
       [0.5548193 , 0.59680611, 0.13818054, 0.27579053, 0.69819252],
       [0.51317158, 0.62152831, 0.11828132, 0.24452002, 0.73746472]])

### Interpretación de las predicciones del ensemble

En esta tabla, cada **columna representa un mismo caso** y cada **fila representa uno de los 10 modelos** del ensemble.

Lo que estamos viendo no son errores ni inconsistencias, sino algo muy valioso:
**qué tan de acuerdo están los modelos entre sí**.

- Cuando los valores de una columna son **muy parecidos**,
  significa que todos los modelos llegan a una conclusión similar.
  En esos casos, la predicción es **estable** y la incertidumbre es baja.

- Cuando los valores de una columna están **más dispersos**,
  significa que los modelos no se ponen completamente de acuerdo.
  Aunque la probabilidad promedio pueda ser alta,
  la variación indica **duda**.

Por ejemplo:
- Hay casos donde todos los modelos predicen alrededor de 0.12–0.14.
  Aquí el acuerdo es fuerte y la predicción es confiable.
- En otros casos, las probabilidades van desde ~0.51 hasta ~0.59.
  Todos se inclinan por la misma clase, pero con distintos niveles de seguridad.

La idea clave es esta:
> la incertidumbre no solo se ve en el valor de la probabilidad,
> sino en cuánto cambia esa probabilidad entre modelos razonables.

Esta variabilidad es una señal directa de **incertidumbre por inestabilidad**,
y nos permite detectar casos que merecen revisión,
incluso cuando el modelo parece seguro.

---

### 2.2) Obtener probabilidades del ensemble (test)
Cada modelo produce una probabilidad.
La colección de esas probabilidades nos permitirá medir variabilidad.

In [11]:
# Probabilidades por modelo
probs_lr_ensemble = np.array([
    model.predict_proba(X_test)[:, 1]
    for model in lr_ensemble
])

### Qué hace este código

1. **Recorre cada modelo del ensemble**
   `for model in lr_ensemble`
   → Va uno por uno por los **10 modelos** que entrenaste.

2. **Cada modelo hace predicciones sobre los mismos datos (`X_test`)**
   `model.predict_proba(X_test)`
   → Devuelve, para cada observación, dos probabilidades:

   * clase 0
   * clase 1

3. **Se queda solo con la probabilidad de la clase 1**
   `[:, 1]`
   → Extrae la probabilidad de interés (clase positiva).

4. **Agrupa todas las predicciones en un solo array**
   `np.array([...])`
   → Junta las salidas de los 10 modelos en una estructura única.

In [12]:
probs_lr_ensemble.shape  # (n_models, n_samples)

(10, 1200)

### Qué significa el resultado

Esto se lee así:

* **10** → número de modelos en el ensemble
* **1200** → número de observaciones en el conjunto de test

En otras palabras:

* cada **fila** corresponde a un modelo distinto,
* cada **columna** corresponde al **mismo caso** evaluado por todos los modelos.

---

## 3) Señales de incertidumbre a partir del ensemble

### ¿Qué vamos a hacer?
A partir de las probabilidades producidas por todos los modelos del ensemble,
vamos a construir dos señales por caso:

- **Probabilidad media** → qué cree el conjunto de modelos.
- **Variabilidad (desviación estándar)** → qué tanto discrepan entre sí.

La combinación de ambas nos permite detectar
casos “seguros” y casos “dudosos”.

### 3.1) Probabilidad media del ensemble

In [13]:
# Media de probabilidades por caso (promedio entre modelos)
mean_prob = probs_lr_ensemble.mean(axis=0)

mean_prob[:5]

array([0.55426818, 0.64168116, 0.12752539, 0.28607541, 0.7235536 ])

### Interpretación — Probabilidad media del ensemble

Cada valor representa la **probabilidad promedio** asignada a un caso
cuando se combinan las predicciones de todos los modelos del ensemble.

En estos ejemplos:
- **0.55** indica que, en promedio, los modelos se inclinan levemente por la clase 1,
  pero sin una convicción fuerte.
- **0.64** muestra una inclinación más clara hacia la clase 1.
- **0.13** indica que el conjunto de modelos coincide en que el caso
  pertenece probablemente a la clase 0.
- **0.29** refleja una baja probabilidad de clase 1,
  aunque no extremadamente baja.
- **0.72** sugiere una probabilidad alta de clase 1,
  cercana a un nivel de decisión.

La probabilidad media resume **qué cree el conjunto de modelos**,
pero por sí sola no indica si esa creencia es estable o dudosa.
Para eso necesitamos observar la variabilidad entre modelos.

### 3.2) Variabilidad del ensemble
Usamos la desviación estándar como medida simple de desacuerdo.

In [14]:
# Desviación estándar por caso
std_prob = probs_lr_ensemble.std(axis=0)

std_prob[:5]

array([0.02344292, 0.03047407, 0.01380539, 0.02459004, 0.02464104])

### Interpretación — Variabilidad del ensemble

Cada valor representa **qué tanto difieren los modelos entre sí**
al predecir la probabilidad de un mismo caso.

En estos ejemplos:
- **0.023** indica que las predicciones de los modelos son muy similares;
  hay buen acuerdo y poca incertidumbre.
- **0.030** muestra una variación ligeramente mayor,
  señal de un desacuerdo moderado entre modelos.
- **0.014** refleja un acuerdo muy fuerte:
  casi todos los modelos opinan lo mismo.
- **0.025** indica una variabilidad baja pero presente.
- **0.025** nuevamente sugiere consenso razonable,
  aunque no perfecto.

En general, valores pequeños de variabilidad significan
que el modelo es **estable** frente a pequeñas variaciones en el entrenamiento.
Valores más altos indican **duda**,
incluso si la probabilidad media parece alta.